In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
##这个版式已更新，只是没有plt show
sns.set_style("ticks")
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def process_U_data(file_path, lat=60, plev=1000):
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    U = ds['U'].mean(dim='lon').sel(plev=plev)
    return U.sel(lat=lat, method='nearest')


def process_T_data(file_path, plev=5000, lat_range=(60, 90)):
    ds = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    T = ds['T'].mean(dim='lon').sel(plev=plev)
    return T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')


def process_U_base(file_path, lat=60, plev=1000, months=None, base_year='0008'):
    ds = xr.open_dataset(file_path)
    months = months or [1,2,3,4,5,6]
    U = ds['U'].sel(plev=plev)
    U = U.sel(time=ds.time.dt.month.isin(months)).mean(dim='lon')
    var = U.sel(lat=lat, method='nearest')
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby('time.dayofyear').mean()
    return ref, clim


def process_T_base(file_path, plev=5000, lat_range=(60,90), months=None, base_year='0008'):
    ds = xr.open_dataset(file_path)
    months = months or [1,2,3,4,5,6]
    T = ds['T'].sel(plev=plev)
    T = T.sel(time=ds.time.dt.month.isin(months)).mean(dim='lon')
    var = T.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    ref = var.sel(time=var.time.dt.year == int(base_year))
    clim = var.groupby('time.dayofyear').mean()
    return ref, clim


def plot_experiments_separate(base_ref_U, base_clim_U, base_ref_T, base_clim_T,
                              experiments, base_year, save_path,
                              lat=60, plev_U=1000, lat_range=(60,90), plev_T=5000):
    """
    Plot U and T in separate figures with unified styling.

    Parameters:
        base_ref_U: 1D array or DataArray of reference zonal wind
        base_clim_U: 1D array or DataArray of climatology zonal wind
        base_ref_T: 1D array or DataArray of reference temperature
        base_clim_T: 1D array or DataArray of climatology temperature
        experiments: list of dicts, each with:
            'U', 'T' DataArrays (with 'member' dim),
            'x_offset', 'label', 'line_color', 'fill_color'
        base_year: string or int for the year label
        save_path: file path prefix (no extension)
        lat: latitude for U label
        plev_U: pressure level (Pa) for U label
        lat_range: tuple for temperature latitude range
        plev_T: pressure level (Pa) for T label

    Returns:
        (fig_U, ax_U), (fig_T, ax_T)
    """
    # Clean up experiment labels
    for exp in experiments:
        lbl = exp['label']
        lbl = lbl.replace('small_pert', 'INT-O3')
        lbl = lbl.replace('NOCOUPL', 'CLIM-3D')
        exp['label'] = lbl

    # --- U figure ---
    fig_U, ax_U = plt.subplots(figsize=(12, 8), constrained_layout=True)
    # Reference and climatology
    ax_U.plot(range(len(base_ref_U)), base_ref_U,
              color='black', linewidth=5, label='Reference')
    ax_U.plot(range(len(base_clim_U)), base_clim_U,
              color='black', linestyle='--', linewidth=5, label='Climatology')
    # Experiments
    for exp in experiments:
        # time axis
        if hasattr(exp['U'], 'time'):
            length = exp['U'].sizes['time']
        else:
            length = len(exp['U'])
        x_vals = np.arange(exp['x_offset'], exp['x_offset'] + length)
        # statistics
        mean_U = exp['U'].mean(dim='member')
        min_U = exp['U'].min(dim='member')
        max_U = exp['U'].max(dim='member')
        ax_U.plot(x_vals, mean_U, linewidth=5,
                  label=exp['label'], color=exp['line_color'])
        ax_U.fill_between(x_vals, min_U, max_U,
                          alpha=0.3, color=exp['fill_color'])

    # Axes styling
    ax_U.set_xlim(0, 150)
    ax_U.set_xticks([0, 31, 59, 90, 120])
    ax_U.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May'], fontsize=16)
    ax_U.tick_params(axis='y', labelsize=16)
    ax_U.set_ylabel(f'Zonal Mean Wind, {lat}\u00b0, {plev_U/100:.0f} hPa (m/s)', fontsize=18)
    # Year textbox
    ax_U.text(0.02, 0.95, f'Year: {base_year}', transform=ax_U.transAxes,
              fontsize=18, verticalalignment='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    # Legend without year
    legend_U = ax_U.legend(fontsize=14)
    # Save
    fig_U.savefig(f'{save_path}_U.pdf', bbox_inches='tight')
    fig_U.savefig(f'{save_path}_U.png', bbox_inches='tight')

    # --- T figure ---
    fig_T, ax_T = plt.subplots(figsize=(12, 8), constrained_layout=True)
    # Reference and climatology
    ax_T.plot(range(len(base_ref_T)), base_ref_T,
              color='black', linewidth=5, label='Reference')
    ax_T.plot(range(len(base_clim_T)), base_clim_T,
              color='black', linestyle='--', linewidth=5, label='Climatology')
    # Experiments
    for exp in experiments:
        if hasattr(exp['T'], 'time'):
            length = exp['T'].sizes['time']
        else:
            length = len(exp['T'])
        x_vals = np.arange(exp['x_offset'], exp['x_offset'] + length)
        mean_T = exp['T'].mean(dim='member')
        min_T = exp['T'].min(dim='member')
        max_T = exp['T'].max(dim='member')
        ax_T.plot(x_vals, mean_T, linewidth=5,
                  label=exp['label'], color=exp['line_color'])
        ax_T.fill_between(x_vals, min_T, max_T,
                          alpha=0.3, color=exp['fill_color'])
    # Chlorine threshold
    ax_T.axhline(197, linestyle='--', linewidth=2, color='royalblue')
    ax_T.text(5, 194, 'Cl activation threshold', fontsize=12,
              color='royalblue', horizontalalignment='left')

    # Axes styling
    ax_T.set_xlim(0, 150)
    ax_T.set_xticks([0, 31, 59, 90, 120])
    ax_T.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May'], fontsize=16)
    ax_T.tick_params(axis='y', labelsize=16)
    ax_T.set_ylabel(f'Min Temp, {lat_range[0]}-{lat_range[1]}\u00b0N, {plev_T/100:.0f} hPa (K)', fontsize=18)
    ax_T.set_xlabel('Day of Year', fontsize=18)
    # Year textbox
    ax_T.text(0.02, 0.95, f'Year: {base_year}', transform=ax_T.transAxes,
              fontsize=18, verticalalignment='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    # Legend without year
    legend_T = ax_T.legend(fontsize=14)

    # Save
    fig_T.savefig(f'{save_path}_T.pdf', bbox_inches='tight')
    fig_T.savefig(f'{save_path}_T.png', bbox_inches='tight')

    return (fig_U, ax_U), (fig_T, ax_T)


# --- 0008 Experiments ---
# Base file path for year 0008
base_file = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'


# 0008 Jan small perturbation (x_offset = 0)
data_0008Jan_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc', lat=60, plev=1000)
data_0008Jan_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb small perturbation (x_offset = 31)
data_0008Feb_small_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc', lat=60, plev=1000)
data_0008Feb_small_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar small perturbation (x_offset = 59)
data_0008Mar_small_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc', lat=60, plev=1000)
data_0008Mar_small_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb large perturbation (x_offset = 31)
data_0008Feb_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc', lat=60, plev=1000)
data_0008Feb_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar large perturbation (x_offset = 59)
data_0008Mar_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc', lat=60, plev=1000)
data_0008Mar_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/*.nc', plev=5000, lat_range=(60,90))

# 0008 Apr large perturbation (x_offset = 89)
data_0008Apr_large_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc', lat=60, plev=1000)
data_0008Apr_large_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb water vapor perturbation (x_offset = 31)
data_0008Feb_vapor_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc', lat=60, plev=1000)
data_0008Feb_vapor_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar water vapor perturbation (x_offset = 59)
data_0008Mar_vapor_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/*.nc', lat=60, plev=1000)
data_0008Mar_vapor_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb NOCOUPL reference (from FNC folder, x_offset = 31)
data_0008Feb_NOCOUPL_ref_U = process_U_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc', lat=60, plev=1000)
data_0008Feb_NOCOUPL_ref_T = process_T_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/*.nc', plev=5000, lat_range=(60,90))

# 0008 Feb NOCOUPL (from nocouple_data folder, x_offset = 31)
data_0008Feb_NOCOUPL_U = process_U_data('/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', lat=60, plev=1000)
data_0008Feb_NOCOUPL_T = process_T_data('/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc', plev=5000, lat_range=(60,90))

# 0008 Mar NOCOUPL (x_offset = 59)
data_0008Mar_NOCOUPL_U = process_U_data('/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', lat=60, plev=1000)
data_0008Mar_NOCOUPL_T = process_T_data('/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc', plev=5000, lat_range=(60,90))

# --- Other Years (0003, 0013, 0014, 0019) for small perturbation ---

def process_year_small(year, month):
    # month should be either 'Feb' or 'Mar'; x_offset = 31 for Feb and 59 for Mar
    offset = 31 if month == 'Feb' else 59
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/{month}/*.nc'
    U_data = process_U_data(base_path, lat=60, plev=1000)
    T_data = process_T_data(base_path, plev=5000, lat_range=(60,90))
    return U_data, T_data, offset

# 0003
data_0003Feb_small_U, data_0003Feb_small_T, off_0003Feb = process_year_small('0003', 'Feb')
data_0003Mar_small_U, data_0003Mar_small_T, off_0003Mar = process_year_small('0003', 'Mar')

# 0013
data_0013Feb_small_U, data_0013Feb_small_T, off_0013Feb = process_year_small('0013', 'Feb')
data_0013Mar_small_U, data_0013Mar_small_T, off_0013Mar = process_year_small('0013', 'Mar')

# 0014
data_0014Feb_small_U, data_0014Feb_small_T, off_0014Feb = process_year_small('0014', 'Feb')
data_0014Mar_small_U, data_0014Mar_small_T, off_0014Mar = process_year_small('0014', 'Mar')

# 0019
data_0019Feb_small_U, data_0019Feb_small_T, off_0019Feb = process_year_small('0019', 'Feb')
data_0019Mar_small_U, data_0019Mar_small_T, off_0019Mar = process_year_small('0019', 'Mar')

# --- NOCOUPL for other years (0013, 0014, 0019) ---

def process_year_nocouple(year, month):
    offset = 31 if month == 'Feb' else 59
    month_code = '02' if month == 'Feb' else '03'
    base_path = f"/home/weiji/restart_exam/nocouple_data/{year}/{month}_NOCOUPL/{year}-{month_code}/*.nc"

    U_data = process_U_data(base_path, lat=60, plev=1000)
    T_data = process_T_data(base_path, plev=5000, lat_range=(60,90))
    return U_data, T_data, offset

# 0013 NOCOUPL
data_0013Feb_NOCOUPL_U, data_0013Feb_NOCOUPL_T, off_0013Feb_noc = process_year_nocouple('0013', 'Feb')
data_0013Mar_NOCOUPL_U, data_0013Mar_NOCOUPL_T, off_0013Mar_noc = process_year_nocouple('0013', 'Mar')

# 0014 NOCOUPL
data_0014Feb_NOCOUPL_U, data_0014Feb_NOCOUPL_T, off_0014Feb_noc = process_year_nocouple('0014', 'Feb')
data_0014Mar_NOCOUPL_U, data_0014Mar_NOCOUPL_T, off_0014Mar_noc = process_year_nocouple('0014', 'Mar')

# 0019 NOCOUPL
data_0019Feb_NOCOUPL_U, data_0019Feb_NOCOUPL_T, off_0019Feb_noc = process_year_nocouple('0019', 'Feb')
data_0019Mar_NOCOUPL_U, data_0019Mar_NOCOUPL_T, off_0019Mar_noc = process_year_nocouple('0019', 'Mar')
# Helper function remains unchanged
def assign_default_colors(exp_list):
    defaults = [('forestgreen', 'forestgreen'), ('royalblue', 'royalblue'), ('hotpink', 'hotpink')]
    for i, exp in enumerate(exp_list):
        if 'line_color' not in exp or not exp['line_color']:
            exp['line_color'] = defaults[i % len(defaults)][0]
        if 'fill_color' not in exp or not exp['fill_color']:
            exp['fill_color'] = defaults[i % len(defaults)][1]
    return exp_list

# Pre-compute base data for each base year to avoid redundant processing
base_years = ["0008", "0003", "0013", "0014", "0019"]
base_data = {}
for year in base_years:
    base_data[year] = {}
    base_data[year]['ref_U'], base_data[year]['clim_U'] = process_U_base(
        base_file, lat=60, plev=1000, months=[1, 2, 3, 4, 5, 6], base_year=year
    )
    base_data[year]['ref_T'], base_data[year]['clim_T'] = process_T_base(
        base_file, plev=5000, lat_range=(60, 90), months=[1, 2, 3, 4, 5, 6], base_year=year
    )

# Plot 1: Validation plot for 0008Feb NOCOUPL_reference and 0008Feb NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_NOCOUPL_reference', 'U': data_0008Feb_NOCOUPL_ref_U, 'T': data_0008Feb_NOCOUPL_ref_T, 'x_offset': 31},
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/Validation_0008Feb_NOCOUPL')

# Plot 2: 0008Jan small, 0008Feb small, 0008Mar small (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Jan_small_pert', 'U': data_0008Jan_U, 'T': data_0008Jan_T, 'x_offset': 0},
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Mar_small_pert', 'U': data_0008Mar_small_U, 'T': data_0008Mar_small_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_small_pert')

# Plot 3: 0008Feb large, 0008Mar large, 0008Apr large (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_large_pert', 'U': data_0008Feb_large_U, 'T': data_0008Feb_large_T, 'x_offset': 31},
    {'label': '0008Mar_large_pert', 'U': data_0008Mar_large_U, 'T': data_0008Mar_large_T, 'x_offset': 59},
    {'label': '0008Apr_large_pert', 'U': data_0008Apr_large_U, 'T': data_0008Apr_large_T, 'x_offset': 89}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_large_pert')

# Plot 4: 0008Feb small, 0008Feb large, 0008Feb water vapor (all February, base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Feb_large_pert', 'U': data_0008Feb_large_U, 'T': data_0008Feb_large_T, 'x_offset': 31},
    {'label': '0008Feb_vapor_pert', 'U': data_0008Feb_vapor_U, 'T': data_0008Feb_vapor_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_Feb_pert_combo')

# Plot 5: 0003Feb small, 0003Mar small (base year: 0003)
base_year = "0003"
exps = [
    {'label': '0003Feb_small_pert', 'U': data_0003Feb_small_U, 'T': data_0003Feb_small_T, 'x_offset': off_0003Feb},
    {'label': '0003Mar_small_pert', 'U': data_0003Mar_small_U, 'T': data_0003Mar_small_T, 'x_offset': off_0003Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0003_small_pert')

# Plot 6: 0013Feb small, 0013Mar small (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_small_pert', 'U': data_0013Feb_small_U, 'T': data_0013Feb_small_T, 'x_offset': off_0013Feb},
    {'label': '0013Mar_small_pert', 'U': data_0013Mar_small_U, 'T': data_0013Mar_small_T, 'x_offset': off_0013Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013_small_pert')

# Plot 7: 0014Feb small, 0014Mar small (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_small_pert', 'U': data_0014Feb_small_U, 'T': data_0014Feb_small_T, 'x_offset': off_0014Feb},
    {'label': '0014Mar_small_pert', 'U': data_0014Mar_small_U, 'T': data_0014Mar_small_T, 'x_offset': off_0014Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014_small_pert')

# Plot 8: 0019Feb small, 0019Mar small (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_small_pert', 'U': data_0019Feb_small_U, 'T': data_0019Feb_small_T, 'x_offset': off_0019Feb},
    {'label': '0019Mar_small_pert', 'U': data_0019Mar_small_U, 'T': data_0019Mar_small_T, 'x_offset': off_0019Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019_small_pert')

# Plot 9: 0008 NOCOUPL experiments (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31},
    {'label': '0008Mar_NOCOUPL', 'U': data_0008Mar_NOCOUPL_U, 'T': data_0008Mar_NOCOUPL_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008_nocouple')

# Plot 10: 0013 NOCOUPL experiments (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_NOCOUPL', 'U': data_0013Feb_NOCOUPL_U, 'T': data_0013Feb_NOCOUPL_T, 'x_offset': off_0013Feb_noc},
    {'label': '0013Mar_NOCOUPL', 'U': data_0013Mar_NOCOUPL_U, 'T': data_0013Mar_NOCOUPL_T, 'x_offset': off_0013Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013_nocouple')

# Plot 11: 0014 NOCOUPL experiments (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_NOCOUPL', 'U': data_0014Feb_NOCOUPL_U, 'T': data_0014Feb_NOCOUPL_T, 'x_offset': off_0014Feb_noc},
    {'label': '0014Mar_NOCOUPL', 'U': data_0014Mar_NOCOUPL_U, 'T': data_0014Mar_NOCOUPL_T, 'x_offset': off_0014Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014_nocouple')

# Plot 12: 0019 NOCOUPL experiments (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_NOCOUPL', 'U': data_0019Feb_NOCOUPL_U, 'T': data_0019Feb_NOCOUPL_T, 'x_offset': off_0019Feb_noc},
    {'label': '0019Mar_NOCOUPL', 'U': data_0019Mar_NOCOUPL_U, 'T': data_0019Mar_NOCOUPL_T, 'x_offset': off_0019Mar_noc}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019_nocouple')

# Plot 13: 0008Feb small vs 0008Feb NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Feb_small_pert', 'U': data_0008Feb_small_U, 'T': data_0008Feb_small_T, 'x_offset': 31},
    {'label': '0008Feb_NOCOUPL', 'U': data_0008Feb_NOCOUPL_U, 'T': data_0008Feb_NOCOUPL_T, 'x_offset': 31}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008Feb_small_vs_nocouple')

# Plot 14: 0008Mar small vs 0008Mar NOCOUPL (base year: 0008)
base_year = "0008"
exps = [
    {'label': '0008Mar_small_pert', 'U': data_0008Mar_small_U, 'T': data_0008Mar_small_T, 'x_offset': 59},
    {'label': '0008Mar_NOCOUPL', 'U': data_0008Mar_NOCOUPL_U, 'T': data_0008Mar_NOCOUPL_T, 'x_offset': 59}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0008Mar_small_vs_nocouple')

# Plot 15: 0013Feb small vs 0013Feb NOCOUPL (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Feb_small_pert', 'U': data_0013Feb_small_U, 'T': data_0013Feb_small_T, 'x_offset': off_0013Feb},
    {'label': '0013Feb_NOCOUPL', 'U': data_0013Feb_NOCOUPL_U, 'T': data_0013Feb_NOCOUPL_T, 'x_offset': off_0013Feb}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013Feb_small_vs_nocouple')

# Plot 16: 0013Mar small vs 0013Mar NOCOUPL (base year: 0013)
base_year = "0013"
exps = [
    {'label': '0013Mar_small_pert', 'U': data_0013Mar_small_U, 'T': data_0013Mar_small_T, 'x_offset': off_0013Mar},
    {'label': '0013Mar_NOCOUPL', 'U': data_0013Mar_NOCOUPL_U, 'T': data_0013Mar_NOCOUPL_T, 'x_offset': off_0013Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0013Mar_small_vs_nocouple')

# Plot 17: 0014Feb small vs 0014Feb NOCOUPL (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Feb_small_pert', 'U': data_0014Feb_small_U, 'T': data_0014Feb_small_T, 'x_offset': off_0014Feb},
    {'label': '0014Feb_NOCOUPL', 'U': data_0014Feb_NOCOUPL_U, 'T': data_0014Feb_NOCOUPL_T, 'x_offset': off_0014Feb}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014Feb_small_vs_nocouple')

# Plot 18: 0014Mar small vs 0014Mar NOCOUPL (base year: 0014)
base_year = "0014"
exps = [
    {'label': '0014Mar_small_pert', 'U': data_0014Mar_small_U, 'T': data_0014Mar_small_T, 'x_offset': off_0014Mar},
    {'label': '0014Mar_NOCOUPL', 'U': data_0014Mar_NOCOUPL_U, 'T': data_0014Mar_NOCOUPL_T, 'x_offset': off_0014Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0014Mar_small_vs_nocouple')

# Plot 19: 0019Feb small vs 0019Feb NOCOUPL (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Feb_small_pert', 'U': data_0019Feb_small_U, 'T': data_0019Feb_small_T, 'x_offset': off_0019Feb},
    {'label': '0019Feb_NOCOUPL', 'U': data_0019Feb_NOCOUPL_U, 'T': data_0019Feb_NOCOUPL_T, 'x_offset': off_0019Feb}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019Feb_small_vs_nocouple')

# Plot 20: 0019Mar small vs 0019Mar NOCOUPL (base year: 0019)
base_year = "0019"
exps = [
    {'label': '0019Mar_small_pert', 'U': data_0019Mar_small_U, 'T': data_0019Mar_small_T, 'x_offset': off_0019Mar},
    {'label': '0019Mar_NOCOUPL', 'U': data_0019Mar_NOCOUPL_U, 'T': data_0019Mar_NOCOUPL_T, 'x_offset': off_0019Mar}
]
exps = assign_default_colors(exps)
plot_experiments_separate(base_data[base_year]['ref_U'], base_data[base_year]['clim_U'],
                    base_data[base_year]['ref_T'], base_data[base_year]['clim_T'],
                    exps, base_year,
                    save_path='/home/weiji/restart_exam/plots/0019Mar_small_vs_nocouple')